# Case Study 01 — Credit Scoring: Exploratory Data Analysis

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Objective

Perform a rigorous exploratory data analysis on the **UCI Default of Credit Card Clients** dataset
(N=30,000, Taiwan, 2005) to:

1. Understand the target variable distribution and class imbalance
2. Identify data quality issues (missing values, outliers, inconsistencies)
3. Analyze univariate and bivariate distributions of key predictors
4. Extract domain-driven hypotheses for the modeling stage

---

## Dataset Reference

| Attribute | Value |
|-----------|-------|
| Source | UCI ML Repository — ID 350 |
| Observations | 30,000 clients |
| Features | 23 predictor variables |
| Target | `default.payment.next.month` (1=default, 0=no default) |
| Period | April–September 2005 |
| Geography | Taiwan (private bank) |

**Citation:** Yeh, I.C. & Lien, C. (2009). *The comparisons of data mining techniques for the predictive accuracy
of probability of default of credit card clients.* Expert Systems with Applications, 36(2), 2473–2480.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

# Add repo root to path so utils/ is importable
REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Institutional plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.edgecolor': '#dcd9d5',
    'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974',
    'ytick.color': '#7a7974',
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'semibold',
    'figure.dpi': 120,
})

TEAL   = '#01696f'
MAROON = '#a12c7b'
GRAY   = '#bab9b4'
GREEN  = '#437a22'
ORANGE = '#964219'

print(f'Python {sys.version}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 1. Data Ingestion

In [ ]:
DATA_DIR = Path('../data')

# Auto-detect file: supports both .xls and .csv formats
candidates = list(DATA_DIR.glob('*.xls')) + list(DATA_DIR.glob('*.xlsx')) + list(DATA_DIR.glob('*.csv'))
if not candidates:
    raise FileNotFoundError(
        'Dataset not found. Run: bash data/download.sh\n'
        'Or download manually from: https://archive.ics.uci.edu/dataset/350'
    )

fpath = candidates[0]
print(f'Loading: {fpath.name}')

if fpath.suffix in ('.xls', '.xlsx'):
    raw = pd.read_excel(fpath, header=1)  # UCI file has 2-row header
else:
    raw = pd.read_csv(fpath)

print(f'Shape: {raw.shape}')
raw.head(3)

In [ ]:
# ── Standardise column names ────────────────────────────────────────────────
df = raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '_')

# Rename target for clarity
target_raw = [c for c in df.columns if 'default' in c][0]
df = df.rename(columns={target_raw: 'default', 'id': 'client_id'})

TARGET = 'default'
ID_COL = 'client_id'
FEATURES = [c for c in df.columns if c not in [TARGET, ID_COL]]

print('Target column:', TARGET)
print('Feature count:', len(FEATURES))
print('Features:', FEATURES)

## 2. Data Quality Assessment

In [ ]:
# ── Missing values ──────────────────────────────────────────────────────────
missing = pd.DataFrame({
    'n_missing': df.isnull().sum(),
    'pct_missing': df.isnull().mean() * 100,
    'dtype': df.dtypes
}).sort_values('n_missing', ascending=False)

print('=== Missing Value Report ===')
print(missing[missing.n_missing > 0].to_string() if missing.n_missing.sum() > 0
      else '✓ No missing values detected.')

In [ ]:
# ── Duplicates ──────────────────────────────────────────────────────────────
n_dupes = df.duplicated(subset=FEATURES).sum()
print(f'Duplicate rows (on features): {n_dupes} ({n_dupes/len(df)*100:.2f}%)')

# ── Data types + cardinality ────────────────────────────────────────────────
profile = pd.DataFrame({
    'dtype': df[FEATURES].dtypes,
    'n_unique': df[FEATURES].nunique(),
    'min': df[FEATURES].min(),
    'max': df[FEATURES].max(),
    'mean': df[FEATURES].mean(),
    'std': df[FEATURES].std(),
})
profile

In [ ]:
# ── Known encoding issues in UCI dataset ────────────────────────────────────
# SEX: 1=male, 2=female (values 0 undocumented)
# EDUCATION: 1=grad school, 2=university, 3=high school, 4=others (0,5,6 undocumented)
# MARRIAGE: 1=married, 2=single, 3=others (0 undocumented)

for col, valid, label in [
    ('sex', [1, 2], 'SEX'),
    ('education', [1, 2, 3, 4], 'EDUCATION'),
    ('marriage', [1, 2, 3], 'MARRIAGE'),
]:
    if col in df.columns:
        invalid = df[~df[col].isin(valid)][col].value_counts()
        if len(invalid):
            print(f'[WARN] {label} — undocumented codes: {invalid.to_dict()}')
        else:
            print(f'[OK]  {label} — all values in valid set')

## 3. Target Variable: Class Imbalance

In [ ]:
target_counts = df[TARGET].value_counts()
default_rate = df[TARGET].mean()

print(f'Default rate   : {default_rate:.2%}')
print(f'Good (0)       : {target_counts[0]:,}')
print(f'Bad  (1)       : {target_counts[1]:,}')
print(f'Good:Bad ratio : {target_counts[0]/target_counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = axes[0].bar(['Good (0)', 'Bad (1)'], target_counts.values,
                   color=[GREEN, MAROON], width=0.5, edgecolor='none')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}\n({val/len(df):.1%})', ha='center', va='bottom', fontsize=9)

# Donut
wedges, _, autotexts = axes[1].pie(
    target_counts.values,
    labels=['Good', 'Bad'],
    colors=[GREEN, MAROON],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.55),
)
axes[1].set_title('Default Rate (Donut)')

plt.tight_layout()
plt.savefig('../reports/01_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved → reports/01_class_distribution.png')

## 4. Demographic Variables

In [ ]:
# ── Default rate by categorical variable ────────────────────────────────────
cat_map = {
    'sex': {1: 'Male', 2: 'Female'},
    'education': {1: 'Grad School', 2: 'University', 3: 'High School', 4: 'Others'},
    'marriage': {1: 'Married', 2: 'Single', 3: 'Others'},
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, mapping) in zip(axes, cat_map.items()):
    tmp = df[df[col].isin(mapping.keys())].copy()
    tmp['label'] = tmp[col].map(mapping)
    dr = tmp.groupby('label')[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(dr.index, dr.values, color=TEAL, edgecolor='none', width=0.5)
    ax.axhline(default_rate, color=MAROON, linestyle='--', lw=1.2, label=f'Overall {default_rate:.1%}')
    ax.set_title(f'Default Rate by {col.upper()}')
    ax.set_ylabel('Default Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    for bar, val in zip(bars, dr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', labelrotation=15)

plt.tight_layout()
plt.savefig('../reports/02_default_rate_demographics.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Credit Limit & Age Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Credit limit distribution
axes[0, 0].hist(df['limit_bal'] / 1000, bins=50, color=TEAL, edgecolor='none', alpha=0.8)
axes[0, 0].set_title('Credit Limit Distribution')
axes[0, 0].set_xlabel('Credit Limit (NT$ thousands)')
axes[0, 0].set_ylabel('Count')

# Credit limit by default
for label, grp in df.groupby(TARGET):
    color = GREEN if label == 0 else MAROON
    name = 'Good' if label == 0 else 'Bad'
    axes[0, 1].hist(grp['limit_bal'] / 1000, bins=50, alpha=0.55,
                   color=color, label=name, edgecolor='none', density=True)
axes[0, 1].set_title('Credit Limit by Default Status')
axes[0, 1].set_xlabel('Credit Limit (NT$ thousands)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend(fontsize=8)

# Age distribution
axes[1, 0].hist(df['age'], bins=40, color=ORANGE, edgecolor='none', alpha=0.8)
axes[1, 0].set_title('Age Distribution')
axes[1, 0].set_xlabel('Age (years)')
axes[1, 0].set_ylabel('Count')

# Default rate by age decile
df['age_decile'] = pd.qcut(df['age'], q=10, labels=False, duplicates='drop') + 1
age_dr = df.groupby('age_decile').agg(
    default_rate=(TARGET, 'mean'),
    age_mid=('age', 'median')
).reset_index()
axes[1, 1].plot(age_dr['age_mid'], age_dr['default_rate'],
               marker='o', color=TEAL, lw=2, markersize=5)
axes[1, 1].axhline(default_rate, color=MAROON, linestyle='--', lw=1.2, label='Overall rate')
axes[1, 1].set_title('Default Rate by Age Decile')
axes[1, 1].set_xlabel('Median Age in Decile')
axes[1, 1].set_ylabel('Default Rate')
axes[1, 1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/03_credit_limit_age.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Payment History (PAY_0 to PAY_6) — Key Risk Driver

In [ ]:
pay_cols = [c for c in df.columns if c.startswith('pay_') and c[4:].isdigit()]
pay_cols = sorted(pay_cols, key=lambda x: int(x.split('_')[1]))[:6]
print('Payment history columns:', pay_cols)

# Default rate by payment status (PAY_0 = most recent month)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

pay_labels = {
    -2: 'No credit', -1: 'Paid duly',
     0: 'Min paid', 1: '1m delay', 2: '2m delay',
     3: '3m+', 4: '4m+', 5: '5m+', 6: '6m+', 7: '7m+', 8: '8m+'
}

for ax, col in zip(axes, pay_cols):
    dr = df.groupby(col)[TARGET].mean().reset_index()
    dr['label'] = dr[col].map(pay_labels).fillna(dr[col].astype(str))
    colors = [MAROON if v >= 1 else TEAL for v in dr[col]]
    bars = ax.bar(dr['label'], dr[TARGET], color=colors, edgecolor='none', width=0.6)
    ax.axhline(default_rate, color=GRAY, linestyle='--', lw=1.2)
    ax.set_title(f'{col.upper()} — Default Rate by Payment Status')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)

plt.tight_layout()
plt.savefig('../reports/04_payment_history_default_rate.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Correlation Matrix — Feature Relationships

In [ ]:
numeric_feats = df[FEATURES + [TARGET]].select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_feats].corr()

# Correlations with target, sorted
target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False)
print('=== Correlation with Target (Pearson) ===')
print(target_corr.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap (top 15 features by abs correlation with target)
top_feats = target_corr.abs().nlargest(14).index.tolist() + [TARGET]
corr_sub = df[top_feats].corr()
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr_sub, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, linecolor='#f3f0ec', ax=axes[0])
axes[0].set_title('Correlation Matrix (Top Features)')
axes[0].tick_params(labelsize=8)

# Bar chart of target correlations
colors_bar = [MAROON if v > 0 else TEAL for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='none')
axes[1].axvline(0, color='#dcd9d5', lw=1)
axes[1].set_title('Correlation with Default Target')
axes[1].set_xlabel('Pearson r')
axes[1].tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('../reports/05_correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Bill Amounts vs. Payment Amounts

In [ ]:
bill_cols = [c for c in df.columns if c.startswith('bill_amt')]
pay_amt_cols = [c for c in df.columns if c.startswith('pay_amt')]

# Utilization ratio: avg bill / credit limit
df['avg_bill'] = df[bill_cols].mean(axis=1)
df['avg_pay_amt'] = df[pay_amt_cols].mean(axis=1)
df['utilization'] = np.clip(df['avg_bill'] / df['limit_bal'].replace(0, np.nan), 0, 5)
df['pay_ratio'] = np.clip(df['avg_pay_amt'] / (df['avg_bill'].replace(0, np.nan)), 0, 5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Utilization by default
for label, grp in df.groupby(TARGET):
    color = GREEN if label == 0 else MAROON
    name = 'Good' if label == 0 else 'Bad'
    axes[0].hist(grp['utilization'].clip(0, 2), bins=50, alpha=0.55,
                color=color, label=name, density=True, edgecolor='none')
axes[0].set_title('Credit Utilization by Default Status')
axes[0].set_xlabel('Utilization Ratio (avg_bill / limit)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)

# Payment ratio by default
for label, grp in df.groupby(TARGET):
    color = GREEN if label == 0 else MAROON
    name = 'Good' if label == 0 else 'Bad'
    axes[1].hist(grp['pay_ratio'].clip(0, 3), bins=50, alpha=0.55,
                color=color, label=name, density=True, edgecolor='none')
axes[1].set_title('Payment Ratio by Default Status')
axes[1].set_xlabel('Payment Ratio (avg_pay_amt / avg_bill)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../reports/06_utilization_payment_ratio.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. EDA Summary & Modeling Hypotheses

In [ ]:
summary = {
    'n_observations': len(df),
    'n_features': len(FEATURES),
    'default_rate': f'{default_rate:.2%}',
    'class_imbalance_ratio': f'{target_counts[0]/target_counts[1]:.1f}:1',
    'avg_credit_limit_NT$': f'{df.limit_bal.mean():,.0f}',
    'avg_age': f'{df.age.mean():.1f} years',
    'missing_values': df.isnull().sum().sum(),
}

print('=== EDA Summary ===')
for k, v in summary.items():
    print(f'  {k:<35} {v}')

print()
print('=== Modeling Hypotheses ===')
hypotheses = [
    'H1: PAY_0 (most recent payment status) is the strongest single predictor.',
    'H2: Higher credit utilization increases default probability.',
    'H3: Clients with graduate education show lower default rates.',
    'H4: Low payment ratio (avg_pay / avg_bill) is a strong delinquency signal.',
    'H5: Age has a non-linear relationship with default — younger and very old clients are riskier.',
]
for h in hypotheses:
    print(f'  {h}')

print()
print('Next notebook: 02_feature_engineering.ipynb')